# 18 — SSAR Compact/Hybrid Pol: Stokes, m-χ, m-δ, m-α

Only runs on the RH/RV branch. Same subset as everything else here — Stokes parameters first, then three variants of the two-component decomposition. RHRH and RVRV are masked the same way as every other term (see Module 13) before the Stokes parameters are computed, so transmit-gap pixels don't feed into g0/g1/g2/g3 or any of the three decompositions.

The complex RHRV term gets a direct magnitude/phase QC check before it feeds into the Stokes parameters, for the same reason Module 14 checks HHHV/HHVV/HVVV: a calibration artifact or transmit-gap residue is visible directly in the channel it came from, not just indirectly in a downstream decomposition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nisar_utils.config import load_config
from nisar_utils.polarimetric_io import load_detected_mode, load_branch_terms, COMPACT_POL_TERMS
from nisar_utils.polarimetry import *
cfg=load_config(); mode=load_detected_mode(cfg)
if mode['polarimetric_mode']!='SSAR_COMPACT_POL': print('SKIPPED:',mode['polarimetric_mode'])
else:
    data,sp,guard,mode=load_branch_terms(cfg,COMPACT_POL_TERMS)
    g0,g1,g2,g3=stokes_from_compact_terms(data['RHRH'],data['RHRV'],data['RVRV'])
    p=compact_child_parameters(g0,g1,g2,g3)
    print('Exact persisted subset:',sp)
    print('m median:',float(np.nanmedian(p['m'])))
    print('delta median (deg):',float(np.nanmedian(np.degrees(p['delta']))))
    print('alpha_s median (deg):',float(np.nanmedian(np.degrees(p['alpha_s']))))
    dchi=m_chi_decomposition(g0,p['m'],p['chi2']); ddelta=m_delta_decomposition(g0,p['m'],p['delta']); dalpha=m_alpha_decomposition(g0,p['m'],p['alpha_s'])
    for name,d in [('m-chi',dchi),('m-delta',ddelta),('m-alpha',dalpha)]:
        r=decomposition_residual(g0,d)
        print(name,'median residual=',float(np.nanmedian(r)),'max abs residual=',float(np.nanmax(np.abs(r))))
    def show(d,title):
        rgb=np.stack([np.sqrt(np.maximum(d['double_bounce'],0)),np.sqrt(np.maximum(d['volume'],0)),np.sqrt(np.maximum(d['surface'],0))],axis=-1)
        rgb/=np.nanpercentile(rgb,99,axis=(0,1),keepdims=True)+1e-12; rgb=np.clip(rgb,0,1)
        plt.figure(figsize=(7,5)); plt.imshow(rgb); plt.title(title+' RGB (R=double, G=volume, B=surface)'); plt.axis('off'); plt.show()
    show(dchi,'m-chi'); show(ddelta,'m-delta'); show(dalpha,'m-alpha')

In [ ]:
from nisar_utils.visualization import plot_complex_term

if mode['polarimetric_mode']=='SSAR_COMPACT_POL':
    fig, (ax_mag, ax_phase) = plt.subplots(1, 2, figsize=(12, 5))
    plot_complex_term(data['RHRV'], 'RHRV', ax_mag=ax_mag, ax_phase=ax_phase)
    fig.suptitle(f'RHRV -- QC before Stokes computation ({mode["polarimetric_frequency"]})')
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    print('RHRV valid (non-NaN) pixels:', int(np.count_nonzero(~np.isnan(np.abs(data["RHRV"])))),
          'of', data['RHRV'].size)